In [3]:
import transformers
from bs4 import BeautifulSoup
import requests
import time
import json
import os
import pandas as pd
from collections import Counter
from contextlib import redirect_stdout, redirect_stderr
import tqdm
import tiktoken
import tokenizers

In [6]:
url = "https://huggingface.co/models?pipeline_tag=text-generation&library=pytorch,transformers&p={}&sort=downloads"
pages = [0, 1, 2, 3, 4]

In [7]:
def scrape_page(page):
    response = requests.get(url.format(page))
    html = BeautifulSoup(response.text)
    models = []
    for model in html.find_all("h4"):
        models.append(model.text.strip())
    return models
all_models = []
for page in pages:
    all_models.extend(scrape_page(page))
    time.sleep(1)
with open("data/hf_text_gen_models.json", "w") as f:
    json.dump(all_models, f, indent=2)

In [54]:
with open("data/hf_text_gen_models.json", "r") as f:
    models = json.load(f)
print(models[:10])

['openai-community/gpt2', 'meta-llama/Llama-3.1-8B-Instruct', 'facebook/opt-125m', 'meta-llama/Llama-3.2-1B-Instruct', 'meta-llama/Llama-3.2-3B-Instruct', 'meta-llama/Meta-Llama-3-8B', 'mistralai/Mistral-7B-Instruct-v0.2', 'EleutherAI/pythia-160m', 'distilbert/distilgpt2', 'openai-community/gpt2-large']


In [55]:
def get_type(tok):
    if hasattr(tok, "tokenizer") and isinstance(tok.tokenizer, tiktoken.core.Encoding):
        return "BPE"
    elif hasattr(tok, "_tokenizer"):
        return type(tok._tokenizer.model).__name__
    elif hasattr(tok, "bpe_ranks") or hasattr(tok, "bpe"):
        return "BPE"
    elif hasattr(tok, "sp_model"):
        return "SP"
    elif isinstance(tok, transformers.models.gpt_neox_japanese.tokenization_gpt_neox_japanese.GPTNeoXJapaneseTokenizer):
        return "BPE"
    

In [56]:
tokenizers = []
errors = []
for model in tqdm.tqdm(models):
    try:        
        with open(os.devnull, 'w') as fnull:
            with redirect_stdout(fnull), redirect_stderr(fnull):
                tok = transformers.AutoTokenizer.from_pretrained(model, trust_remote_code=True)
        tokenizers.append((model, get_type(tok)))
    except Exception as e:
        print(f"Error loading {model}: {e}")
        errors.append(model)
df = pd.DataFrame(tokenizers, columns=["model", "tokenizer_type"])
df.to_csv("data/hf_text_gen_tokenizers.csv", index=False)
print(df.head())

 31%|███████████████████████████████████████▍                                                                                      | 47/150 [00:28<00:46,  2.21it/s]

Error loading Rostlab/prot_t5_xl_bfd: You're trying to run a `Unigram` model but you're file was trained with a different algorithm


 33%|█████████████████████████████████████████▏                                                                                    | 49/150 [00:29<00:45,  2.21it/s]

Error loading lmstudio-community/NVIDIA-Nemotron-3-Nano-30B-A3B-MLX-4bit: Tokenizer class TokenizersBackend does not exist or is not currently imported.


 34%|██████████████████████████████████████████▊                                                                                   | 51/150 [00:30<00:35,  2.82it/s]

...

Error loading yuhuili/EAGLE-LLaMA3-Instruct-8B: not a string


 77%|███████████████████████████████████████████████████████████████████████████████████████████████▊                             | 115/150 [01:09<00:28,  1.24it/s]

Error loading SanctumAI/Meta-Llama-3.1-8B-Instruct-GGUF: not a string


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 150/150 [01:31<00:00,  1.65it/s]

                              model tokenizer_type
0             openai-community/gpt2            BPE
1  meta-llama/Llama-3.1-8B-Instruct            BPE
2                 facebook/opt-125m            BPE
3  meta-llama/Llama-3.2-1B-Instruct            BPE
4  meta-llama/Llama-3.2-3B-Instruct            BPE


In [57]:
df = pd.read_csv("data/hf_text_gen_tokenizers.csv")
tokenizer_counts = Counter(df.iloc[:]["tokenizer_type"])
tokenizer_counts

Counter({'BPE': 130, 'Unigram': 5, 'SP': 4, nan: 1, 'WordPiece': 1})